# k09 — D29: curve FID and tuning sensitivity analysis (Run 1, Kaggle T4)

Declared in `DECISION_LOG.md` D29 **before** this notebook was run.

* **Part 1 (D29 item 4):** FID for the 1.0, 2.0, 3.0, 4.0 GPU-hour checkpoints of both arms, from the stored curve images (500 per class).
* **Part 2 (D29 item 6):** classifier tuning, identical for A, B, C, D. Six settings (learning rate 0.00003 / 0.0001 / 0.0003 x unweighted / class-weighted loss). Choice by **validation** macro-F1 with seed 42; the chosen setting is then run with seeds 43 and 44. The test set is never used to choose.
* **Part 3 (D29 item 5, optional):** Condition D without synthetic melanoma images, declared settings, three seeds.

Set `GIT_HASH` in the first cell to the commit that contains D29 and the classifier patch. Run once with `SMOKE = True`, then `SMOKE = False` with Save & Run All.

In [1]:
GIT_HASH = "8e5f6c0"     # commit with D29 + patch_classifier_d29 (git log -1 --format=%h)
SMOKE    = False                       # True: quick test (minutes). False: the real run (about 6.5 GPU-h)
assert GIT_HASH != "PUT_COMMIT_HASH_HERE", "set GIT_HASH first"

import kagglehub, os, glob, hashlib
IMG  = kagglehub.dataset_download("ashwanisingh997/ham10000-images-1229692")
SPL  = kagglehub.dataset_download("ashwanisingh997/ham10000-splits-1229692")
CODE = kagglehub.dataset_download("ashwanisingh997/thesis-code-1229692")
SYN  = kagglehub.dataset_download("ashwanisingh997/run1-synthetic-headline-1229692")
CA   = kagglehub.dataset_download("ashwanisingh997/run1-curve-a-1229692")
CB   = kagglehub.dataset_download("ashwanisingh997/run1-curve-b-1229692")

h = hashlib.sha256(open(f"{CODE}/metrics.py", "rb").read()).hexdigest().upper()
assert h == "602747135A78568EAF96E64DB70A7C43CAC6F8527089DBE26711B0D834F05E17", "metrics.py changed"
clf = open(f"{CODE}/classifier.py").read()
assert "class_weights=None" in clf and "class_weights is None" in clf, "code dataset is not v6 (D29 patch missing)"
print("code: metrics.py unchanged; classifier.py has the D29 class_weights option")

SYN_ROOT = f"{SYN}/synthetic/run1_cloud"
for arm in ["gan_final", "lora_final"]:
    for c in ["akiec", "bcc", "df", "mel", "vasc"]:
        assert len(glob.glob(f"{SYN_ROOT}/{arm}/{c}/*.png")) == 1000, (arm, c)
LABELS = ["1.0h", "2.0h", "3.0h", "4.0h"]
ROOTS  = {"1.0h": f"{CA}/curve_a", "2.0h": f"{CA}/curve_a", "3.0h": f"{CB}/curve_b", "4.0h": f"{CB}/curve_b"}
for lab in LABELS:
    for arm in ["gan", "lora"]:
        for c in ["akiec", "bcc", "df", "mel", "vasc"]:
            assert len(glob.glob(f"{ROOTS[lab]}/curve_{lab}/{arm}/{c}/*.png")) == 500, (lab, arm, c)
print("headline 10,000 and curve 20,000 images confirmed")

code: metrics.py unchanged; classifier.py has the D29 class_weights option
headline 10,000 and curve 20,000 images confirmed


In [2]:
!pip install -q --no-deps pytorch-fid
import pytorch_fid
print("pytorch-fid OK")

pytorch-fid OK


In [3]:
import sys, shutil, json, gc, torch, numpy as np, pandas as pd
WORK = "/kaggle/working/repo"
os.makedirs(f"{WORK}/src", exist_ok=True); os.makedirs(f"{WORK}/config", exist_ok=True)
for f in os.listdir(CODE):
    shutil.copy(f"{CODE}/{f}", f"{WORK}/config/{f}" if f.endswith(".yaml") else f"{WORK}/src/{f}")
sys.path.insert(0, WORK); os.chdir(WORK)
print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
assert torch.__version__.startswith("2.11")

from src.run_registry import init_run
root = init_run("run1_cloud", root="/kaggle/working/results", git_hash=GIT_HASH)
from src.data import (HAM10000, make_loader, build_image_index, assert_no_synthetic_leakage,
                      generator_source_frame, AUG_CLASSES, CLASSES, C2I)
assert_no_synthetic_leakage(f"{SPL}/val.csv", f"{SPL}/test.csv")
index = build_image_index(IMG)
NW = 4
val_ds  = HAM10000(f"{SPL}/val.csv",  IMG, size=128, augment=False, index=index)
test_ds = HAM10000(f"{SPL}/test.csv", IMG, size=128, augment=False, index=index)
val_loader  = make_loader(val_ds,  batch_size=32, shuffle=False, num_workers=NW)
test_loader = make_loader(test_ds, batch_size=32, shuffle=False, num_workers=NW)
PRE = "SMOKE_" if SMOKE else ""
print(f"val {len(val_ds)}  test {len(test_ds)}  | prefix '{PRE}'")

2.11.0+cu130 | GPU: Tesla T4
[registry] run1_cloud (PRIMARY - Kaggle free-tier GPU, matches proposal Table 8.2)
[registry] segment 1 on Tesla T4 | torch 2.11.0+cu130 | cuda 13.0
[registry] writing to /kaggle/working/results/run1_cloud
[data] no synthetic leakage in 2 partition(s)
[data] indexed 10015 images under /kaggle/input/datasets/ashwanisingh997/ham10000-images-1229692
val 1532  test 1502  | prefix ''


## Part 1 — FID at each efficiency-curve checkpoint (D29 item 4)

In [4]:
from src.fid import InceptionFeatures, fid
frame = generator_source_frame(f"{SPL}/train.csv")
ref_paths = [index[i] for i in frame.image_id]
assert len(ref_paths) == 1526

NCURVE = 50 if SMOKE else 500
LABS_FID = ["1.0h"] if SMOKE else LABELS
inc = InceptionFeatures()
real = {c: inc([p for p, d in zip(ref_paths, frame.dx) if d == c]) for c in AUG_CLASSES}
rows = []
for lab in LABS_FID:
    for arm in ["gan", "lora"]:
        syn = {c: inc(sorted(glob.glob(f"{ROOTS[lab]}/curve_{lab}/{arm}/{c}/*.png"))[:NCURVE]) for c in AUG_CLASSES}
        for c in AUG_CLASSES:
            rows.append({"checkpoint": lab, "arm": arm, "class": c, "n_real": len(real[c]),
                         "n_syn": len(syn[c]), "fid": fid(real[c], syn[c])})
        rows.append({"checkpoint": lab, "arm": arm, "class": "ALL", "n_real": 1526,
                     "n_syn": sum(len(v) for v in syn.values()),
                     "fid": fid(np.concatenate(list(real.values())), np.concatenate(list(syn.values())))})
        print(lab, arm, "pooled FID", round(rows[-1]["fid"], 2))
fid_curve = pd.DataFrame(rows)
fid_curve.to_csv(f"{root}/metrics/{PRE}fid_curve.csv", index=False)
print(fid_curve[fid_curve["class"] == "ALL"].pivot(index="checkpoint", columns="arm", values="fid").round(2).to_string())
del inc; gc.collect(); torch.cuda.empty_cache()

[data] generator source frame: 1526 images across ['akiec', 'bcc', 'df', 'mel', 'vasc']
dx
mel      773
bcc      361
akiec    222
vasc      99
df        71
Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth


100%|██████████| 91.2M/91.2M [00:01<00:00, 70.2MB/s]


1.0h gan pooled FID 207.37
1.0h lora pooled FID 121.4
2.0h gan pooled FID 204.87
2.0h lora pooled FID 135.47
3.0h gan pooled FID 193.67
3.0h lora pooled FID 94.74
4.0h gan pooled FID 209.57
4.0h lora pooled FID 138.82
arm            gan    lora
checkpoint                
1.0h        207.37  121.40
2.0h        204.87  135.47
3.0h        193.67   94.74
4.0h        209.57  138.82


## Part 2 — Tuning sensitivity, step 1: choose a setting per condition on validation (seed 42)

In [5]:
from src.utils import set_seed
from src.classifier import build_model, train_classifier
from src.metrics import evaluate, save_metrics

def synthetic_rows(arm, classes=AUG_CLASSES, n=1000):
    rows = []
    for c in classes:
        files = sorted(glob.glob(f"{SYN_ROOT}/{arm}/{c}/*.png"))[:n]
        assert len(files) == n, (arm, c, len(files))
        rows += [{"image_id": os.path.basename(f)[:-4], "dx": c, "path": f, "synthetic": True} for f in files]
    return pd.DataFrame(rows)

CONDS   = ["A", "B", "C", "D"]
EXTRA   = {"A": None, "B": None, "C": synthetic_rows("gan_final"), "D": synthetic_rows("lora_final")}
AUGMENT = {"A": False, "B": True, "C": False, "D": False}
EXPECT  = {"A": 6981, "B": 6981, "C": 11981, "D": 11981}

def train_set(cond, extra=None):
    tr = HAM10000(f"{SPL}/train.csv", IMG, size=128, augment=AUGMENT.get(cond, False), index=index,
                  extra_rows=EXTRA[cond] if extra is None else extra)
    return tr

def inv_freq_weights(tr):
    n = tr.df.dx.map(C2I).value_counts().reindex(range(len(CLASSES)), fill_value=0).values.astype(float)
    assert (n > 0).all(), n
    return (n.sum() / (len(CLASSES) * n)).tolist()

LRS    = [1e-4] if SMOKE else [3e-5, 1e-4, 3e-4]
LOSSES = ["unweighted", "weighted"]
EPOCHS = 1 if SMOKE else 30
SETTINGS = [(lr, loss) for lr in LRS for loss in LOSSES]
CKDIR = "/tmp/d29_models"; os.makedirs(CKDIR, exist_ok=True)   # seed-42 models kept outside the outputs

sel_rows = []
for cond in CONDS:
    tr = train_set(cond); assert len(tr) == EXPECT[cond], (cond, len(tr))
    w = inv_freq_weights(tr)
    for lr, loss in SETTINGS:
        name = f"{PRE}TUNE_{cond}_lr{lr:g}_{loss}_seed42"
        print(f"\n{'='*60}\n{name}\n{'='*60}")
        set_seed(42)
        ld = make_loader(tr, batch_size=32, shuffle=True, num_workers=NW)
        m, hist = train_classifier(build_model(), ld, val_loader, epochs=EPOCHS, lr=lr, weight_decay=1e-4,
                                   patience=7, tag=name, log_dir=f"{root}/logs",
                                   class_weights=w if loss == "weighted" else None)
        best_val = max(h["val_macro_f1"] for h in hist)
        torch.save(m.state_dict(), f"{CKDIR}/{name}.pt")
        sel_rows.append({"condition": cond, "lr": lr, "loss": loss, "val_macro_f1": best_val,
                         "epochs_run": len(hist), "class_weights": [round(x, 3) for x in w] if loss == "weighted" else None})
        del m; torch.cuda.empty_cache()

sel = pd.DataFrame(sel_rows)
sel["chosen"] = False
for cond in CONDS:
    i = sel[sel.condition == cond].val_macro_f1.idxmax()
    sel.loc[i, "chosen"] = True
sel.to_csv(f"{root}/metrics/{PRE}tuning_selection.csv", index=False)
print(sel.round(4).to_string(index=False))
CHOSEN = {r.condition: (r.lr, r.loss) for r in sel[sel.chosen].itertuples()}
print("\nchosen on VALIDATION macro-F1:", CHOSEN)


TUNE_A_lr3e-05_unweighted_seed42
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 207MB/s]


ep 00  loss 1.0512  val macroF1 0.2452  val acc 0.7245
ep 01  loss 0.6354  val macroF1 0.4658  val acc 0.7755
ep 02  loss 0.4208  val macroF1 0.5037  val acc 0.7859
ep 03  loss 0.2609  val macroF1 0.5218  val acc 0.7794
ep 04  loss 0.1603  val macroF1 0.5255  val acc 0.7866
ep 05  loss 0.0955  val macroF1 0.5304  val acc 0.7872
ep 06  loss 0.0568  val macroF1 0.5646  val acc 0.7833
ep 07  loss 0.0404  val macroF1 0.5699  val acc 0.7826
ep 08  loss 0.0347  val macroF1 0.5711  val acc 0.7846
ep 09  loss 0.0244  val macroF1 0.5761  val acc 0.7924
ep 10  loss 0.0216  val macroF1 0.5685  val acc 0.7846
ep 11  loss 0.0192  val macroF1 0.5807  val acc 0.7996
ep 12  loss 0.0161  val macroF1 0.5629  val acc 0.7852
ep 13  loss 0.0157  val macroF1 0.5623  val acc 0.7813
ep 14  loss 0.0154  val macroF1 0.5808  val acc 0.7911
ep 15  loss 0.0112  val macroF1 0.5729  val acc 0.7833
ep 16  loss 0.0081  val macroF1 0.5683  val acc 0.7905
ep 17  loss 0.0069  val macroF1 0.5839  val acc 0.7944
ep 18  los

## Part 2 — step 2: chosen setting, seeds 42 (stored model), 43 and 44, evaluated on the test set

In [6]:
SEEDS2 = [42, 43] if SMOKE else [42, 43, 44]
for cond in CONDS:
    lr, loss = CHOSEN[cond]
    tr = train_set(cond); w = inv_freq_weights(tr)
    for seed in SEEDS2:
        name = f"{PRE}TUNED_{cond}_seed{seed}"
        if os.path.exists(f"{root}/metrics/{name}.json"):
            print(f"skip {name}"); continue
        m = build_model()
        if seed == 42:   # the model already trained in step 1 with this setting
            m.load_state_dict(torch.load(f"{CKDIR}/{PRE}TUNE_{cond}_lr{lr:g}_{loss}_seed42.pt"))
            m.to("cuda")
        else:
            print(f"\n{'='*60}\n{name}  (lr {lr:g}, {loss})\n{'='*60}")
            set_seed(seed)
            ld = make_loader(tr, batch_size=32, shuffle=True, num_workers=NW)
            m, _ = train_classifier(m, ld, val_loader, epochs=EPOCHS, lr=lr, weight_decay=1e-4,
                                    patience=7, tag=name, log_dir=f"{root}/logs",
                                    class_weights=w if loss == "weighted" else None)
        r = evaluate(m, test_loader, bootstrap=True, seed=seed)
        r["d29_setting"] = {"lr": lr, "loss": loss}
        save_metrics(r, name, root=str(root))
        del m; torch.cuda.empty_cache()

rows = []
for cond in CONDS:
    rs = [json.load(open(f"{root}/metrics/{PRE}TUNED_{cond}_seed{s}.json")) for s in SEEDS2]
    for metric in ["macro_f1", "melanoma_recall", "balanced_accuracy", "macro_auroc", "macro_auprc"]:
        v = [r[metric] for r in rs]
        rows.append({"condition": cond, "setting": f"lr {CHOSEN[cond][0]:g}, {CHOSEN[cond][1]}", "metric": metric,
                     "mean": np.mean(v), "sd": np.std(v, ddof=1), "values": [round(x, 4) for x in v]})
tuned = pd.DataFrame(rows).round(4)
tuned.to_csv(f"{root}/metrics/{PRE}TUNED_summary.csv", index=False)
print(tuned.pivot(index="metric", columns="condition", values="mean").to_string())

              precision    recall  f1-score   support

       akiec      0.600     0.635     0.617        52
         bcc      0.721     0.690     0.705        71
         bkl      0.678     0.617     0.646       167
          df      0.533     0.400     0.457        20
         mel      0.610     0.515     0.558       167
          nv      0.890     0.933     0.911      1004
        vasc      0.889     0.762     0.821        21

    accuracy                          0.820      1502
   macro avg      0.703     0.650     0.674      1502
weighted avg      0.812     0.820     0.815      1502

macro-F1 0.6735 | balanced acc 0.6502 | MEL recall 0.5150 | accuracy 0.8202 | macro-AUROC 0.9464

TUNED_A_seed43  (lr 0.0003, weighted)
ep 00  loss 1.2380  val macroF1 0.4425  val acc 0.6247
ep 01  loss 0.7353  val macroF1 0.5314  val acc 0.7023
ep 02  loss 0.4614  val macroF1 0.5661  val acc 0.6926
ep 03  loss 0.2876  val macroF1 0.5303  val acc 0.6847
ep 04  loss 0.2589  val macroF1 0.6191  val acc

## Part 3 (optional, D29 item 5) — Condition D without synthetic melanoma, declared settings

In [7]:
RUN_D_NO_MEL = True
if RUN_D_NO_MEL:
    extra = synthetic_rows("lora_final", classes=[c for c in AUG_CLASSES if c != "mel"])
    assert len(extra) == 4000
    for seed in ([42] if SMOKE else [42, 43, 44]):
        name = f"{PRE}D_diffusion_noMEL_seed{seed}"
        if os.path.exists(f"{root}/metrics/{name}.json"):
            print(f"skip {name}"); continue
        print(f"\n{'='*60}\n{name}\n{'='*60}")
        set_seed(seed)
        tr = HAM10000(f"{SPL}/train.csv", IMG, size=128, augment=False, index=index, extra_rows=extra)
        assert len(tr) == 6981 + 4000
        ld = make_loader(tr, batch_size=32, shuffle=True, num_workers=NW)
        m, _ = train_classifier(build_model(), ld, val_loader, epochs=EPOCHS, lr=1e-4, weight_decay=1e-4,
                                patience=7, tag=name, log_dir=f"{root}/logs")
        save_metrics(evaluate(m, test_loader, bootstrap=True, seed=seed), name, root=str(root))
        del m; torch.cuda.empty_cache()
    print("D without MEL complete")


D_diffusion_noMEL_seed42
ep 00  loss 0.5700  val macroF1 0.5669  val acc 0.7892
ep 01  loss 0.2385  val macroF1 0.6168  val acc 0.8016
ep 02  loss 0.1142  val macroF1 0.6085  val acc 0.8172
ep 03  loss 0.0624  val macroF1 0.5624  val acc 0.7918
ep 04  loss 0.0465  val macroF1 0.5906  val acc 0.8074
ep 05  loss 0.0304  val macroF1 0.5791  val acc 0.8009
ep 06  loss 0.0365  val macroF1 0.5718  val acc 0.7996
ep 07  loss 0.0201  val macroF1 0.6216  val acc 0.8055
ep 08  loss 0.0188  val macroF1 0.5422  val acc 0.7990
ep 09  loss 0.0225  val macroF1 0.5899  val acc 0.7892
ep 10  loss 0.0137  val macroF1 0.6334  val acc 0.8003
ep 11  loss 0.0139  val macroF1 0.6338  val acc 0.8120
ep 12  loss 0.0105  val macroF1 0.6104  val acc 0.7990
ep 13  loss 0.0072  val macroF1 0.6100  val acc 0.7996
ep 14  loss 0.0098  val macroF1 0.5907  val acc 0.7950
ep 15  loss 0.0073  val macroF1 0.6347  val acc 0.8127
ep 16  loss 0.0036  val macroF1 0.6133  val acc 0.8094
ep 17  loss 0.0014  val macroF1 0.6550 

In [8]:
n_npz = len(glob.glob(f"{root}/metrics/{PRE}TUNED_*_predictions.npz"))
assert n_npz == len(CONDS) * len(SEEDS2), n_npz
print(f"{n_npz} tuned prediction files")
shutil.make_archive(f"/kaggle/working/{PRE}run1_d29_results", "zip", "/kaggle/working/results")
print(f"saved: /kaggle/working/{PRE}run1_d29_results.zip")

12 tuned prediction files
saved: /kaggle/working/run1_d29_results.zip
